In [1]:
import random

from distributed_processing.serializers import DummySerializer
from distributed_processing.client import Client
from distributed_processing.filesystem_connector import FileSystemConnector


In [2]:
CURRO = True

if CURRO:
    NS_PATH ="G:\\fs_namespaces\\prueba_distribuida"
    NS_PATH ="C:\\fs_namespaces\\prueba_distribuida"

else:
    NS_PATH = "/home/augusto/python/notebooks/fs_namespaces/prueba_distribuida"


In [3]:
import logging
logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
#logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")

2026-02-24T16:08:21.010166 registry_lock set in C:\fs_namespaces\prueba_distribuida\ud_variables
2026-02-24T16:08:21.046005 registry_lock unset in C:\fs_namespaces\prueba_distribuida\ud_variables
2026-02-24T16:08:21.047677 nclients_lock set in C:\fs_namespaces\prueba_distribuida\ud_variables


2026-02-24T16:08:21.067441 nclients_lock unset in C:\fs_namespaces\prueba_distribuida\ud_variables
Client with id: fs_client_1
Results queue: fs_client_1_responses
2026-02-24T16:08:21.069343 Client: fs_client_1 with responses queue: fs_client_1_responses connected


In [ ]:
client.update_registry_cache()

2026-02-24T16:08:21.078578 registry_lock set in C:\fs_namespaces\prueba_distribuida\ud_variables
2026-02-24T16:08:21.091184 registry_lock unset in C:\fs_namespaces\prueba_distribuida\ud_variables


{'add': ['requests_cola_1'],
 'dic': ['requests_cola_1'],
 'div': ['requests_cola_1'],
 'eval_py_function': ['requests_py_eval'],
 'info': ['requests_fs_server_1'],
 'lista': ['requests_cola_1'],
 'mul': ['requests_cola_1'],
 'sleep': ['requests_cola_1'],
 'tupla': ['requests_cola_1']}

In [6]:
y = client.rpc_async("add", [1, 0])

2026-02-24T16:08:21.124608 Client: fs_client_1 sent request with id: fs_client_1:1 to queue: requests_cola_1


In [7]:
y.get()

2026-02-24T16:08:21.272379 Client: fs_client_1 received a Single RESULT with id: fs_client_1:1


1

In [8]:
client.check_registry ="always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

'method_queues_fake'


In [9]:
client.check_registry ="never" # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


2026-02-24T16:08:21.332858 Client: fs_client_1 sent request with id: fs_client_1:2 to queue: requests_cola_1
2026-02-24T16:08:21.419155 Client: fs_client_1 received a Single ERROR with id: fs_client_1:2


Error -32601 : The method does not exist/is not available.

 


In [10]:
client.check_registry ="never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

2026-02-24T16:08:21.448616 Client: fs_client_1 sent request with id: fs_client_1:3 to queue: requests_cola_1
2026-02-24T16:08:21.662981 Client: fs_client_1 received a Single ERROR with id: fs_client_1:3


'Esto es un error'

In [11]:
client.check_registry ="cache"

In [12]:
x = client.rpc_async("div", [1, 0])

2026-02-24T16:08:21.720789 Client: fs_client_1 sent request with id: fs_client_1:4 to queue: requests_cola_1


In [13]:
try:
    x.get()
except Exception as e:
    print(e)

2026-02-24T16:08:21.869404 Client: fs_client_1 received a Single ERROR with id: fs_client_1:4


Error -3260 : Undefined error

 


In [14]:
client.rpc_sync("add", [1, 1])

2026-02-24T16:08:21.916236 Client: fs_client_1 sent request with id: fs_client_1:5 to queue: requests_cola_1
2026-02-24T16:08:22.155958 Client: fs_client_1 received a Single RESULT with id: fs_client_1:5


2

In [15]:
def f(x,y): return x + y

y = client.rpc_async_fn(f, [1, 2.0])

2026-02-24T16:08:22.196861 Client: fs_client_1 sent request with id: fs_client_1:6 to queue: requests_py_eval


In [16]:
y.get()

2026-02-24T16:08:22.363197 Client: fs_client_1 received a Single RESULT with id: fs_client_1:6


3.0

In [17]:
client.rpc_sync_fn(f, [3.0, 2.0])

2026-02-24T16:08:22.393640 Client: fs_client_1 sent request with id: fs_client_1:7 to queue: requests_py_eval
2026-02-24T16:08:22.579784 Client: fs_client_1 received a Single RESULT with id: fs_client_1:7


5.0

In [18]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)
    fs.append(client.rpc_async(t[0], t[1]))

t = ("sleep", [10], {})
tp.append(t)
fs.append(client.rpc_async(t[0], t[1]))



2026-02-24T16:08:22.624496 Client: fs_client_1 sent request with id: fs_client_1:8 to queue: requests_cola_1
2026-02-24T16:08:22.652675 Client: fs_client_1 sent request with id: fs_client_1:9 to queue: requests_cola_1
2026-02-24T16:08:22.671301 Client: fs_client_1 sent request with id: fs_client_1:10 to queue: requests_cola_1
2026-02-24T16:08:22.703542 Client: fs_client_1 sent request with id: fs_client_1:11 to queue: requests_cola_1
2026-02-24T16:08:22.730425 Client: fs_client_1 sent request with id: fs_client_1:12 to queue: requests_cola_1
2026-02-24T16:08:22.750688 Client: fs_client_1 sent request with id: fs_client_1:13 to queue: requests_cola_1
2026-02-24T16:08:22.769438 Client: fs_client_1 sent request with id: fs_client_1:14 to queue: requests_cola_1
2026-02-24T16:08:22.788653 Client: fs_client_1 sent request with id: fs_client_1:15 to queue: requests_cola_1


('add', [0.9036116114294063, 0.07305889603633309], {})
('add', [0.09742601458133926, 0.4900711273129885], {})
('div', [0.7207228177169482, 0.6747504242200711], {})
('lista', [0.30252028666243946, 0.009397501679387243], {})
('lista', [0.10256825028452432, 0.23266945623182167], {})
('dic', [0.19242795927761092, 0.4842972791709561], {})
('lista', [0.46350235878312995, 0.972627751404377], {})
('lista', [0.24570457710761962, 0.553547538112171], {})
('tupla', [0.77552703587268, 0.2392960333276557], {})


2026-02-24T16:08:22.806713 Client: fs_client_1 sent request with id: fs_client_1:16 to queue: requests_cola_1
2026-02-24T16:08:22.832382 Client: fs_client_1 sent request with id: fs_client_1:17 to queue: requests_cola_1
2026-02-24T16:08:22.853228 Client: fs_client_1 sent request with id: fs_client_1:18 to queue: requests_cola_1


('div', [0.15833083320378927, 0.29776162457617084], {})


In [19]:
resultados = [f.get() for f in fs]
resultados

2026-02-24T16:08:22.910478 Client: fs_client_1 received a Single RESULT with id: fs_client_1:8
2026-02-24T16:08:22.911550 Client: fs_client_1 received a Single RESULT with id: fs_client_1:9
2026-02-24T16:08:22.912020 Client: fs_client_1 received a Single RESULT with id: fs_client_1:10
2026-02-24T16:08:22.912474 Client: fs_client_1 received a Single RESULT with id: fs_client_1:11
2026-02-24T16:08:22.966764 Client: fs_client_1 received a Single RESULT with id: fs_client_1:12
2026-02-24T16:08:23.049608 Client: fs_client_1 received a Single RESULT with id: fs_client_1:13
2026-02-24T16:08:23.151775 Client: fs_client_1 received a Single RESULT with id: fs_client_1:14
2026-02-24T16:08:23.168572 Client: fs_client_1 received a Single RESULT with id: fs_client_1:15
2026-02-24T16:08:23.312912 Client: fs_client_1 received a Single RESULT with id: fs_client_1:16
2026-02-24T16:08:23.327182 Client: fs_client_1 received a Single RESULT with id: fs_client_1:17


2026-02-24T16:08:33.353214 Client: fs_client_1 received a Single RESULT with id: fs_client_1:18


[0.9766705074657394,
 0.5874971418943278,
 1.0681324410429465,
 [0.30252028666243946, 0.009397501679387243],
 [0.10256825028452432, 0.23266945623182167],
 {'a': 0.19242795927761092, 'b': [0.19242795927761092, 0.4842972791709561]},
 [0.46350235878312995, 0.972627751404377],
 [0.24570457710761962, 0.553547538112171],
 (0.77552703587268, 0.2392960333276557),
 0.5317368664587147,
 None]

In [20]:
fs = client.rpc_multi_async(tp)

2026-02-24T16:08:33.384073 Client: fs_client_1 sent request with id: fs_client_1:19 to queue: requests_cola_1


2026-02-24T16:08:33.404361 Client: fs_client_1 sent request with id: fs_client_1:20 to queue: requests_cola_1
2026-02-24T16:08:33.430597 Client: fs_client_1 sent request with id: fs_client_1:21 to queue: requests_cola_1
2026-02-24T16:08:33.449886 Client: fs_client_1 sent request with id: fs_client_1:22 to queue: requests_cola_1
2026-02-24T16:08:33.467621 Client: fs_client_1 sent request with id: fs_client_1:23 to queue: requests_cola_1
2026-02-24T16:08:33.489311 Client: fs_client_1 sent request with id: fs_client_1:24 to queue: requests_cola_1
2026-02-24T16:08:33.509334 Client: fs_client_1 sent request with id: fs_client_1:25 to queue: requests_cola_1
2026-02-24T16:08:33.534433 Client: fs_client_1 sent request with id: fs_client_1:26 to queue: requests_cola_1
2026-02-24T16:08:33.564648 Client: fs_client_1 sent request with id: fs_client_1:27 to queue: requests_cola_1
2026-02-24T16:08:33.592979 Client: fs_client_1 sent request with id: fs_client_1:28 to queue: requests_cola_1
2026-02-24

In [21]:
# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory. 
# If there are responses available in the queue or in the cache 
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

2026-02-24T16:08:33.667900 Client: fs_client_1 received a Single RESULT with id: fs_client_1:19
2026-02-24T16:08:33.669098 Client: fs_client_1 received a Single RESULT with id: fs_client_1:20
2026-02-24T16:08:33.670231 Client: fs_client_1 received a Single RESULT with id: fs_client_1:21
2026-02-24T16:08:33.685739 Client: fs_client_1 received a Single RESULT with id: fs_client_1:22


['OK',
 'OK',
 'OK',
 'OK',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING']

In [22]:
client.wait_responses(timeout=2)

2026-02-24T16:08:33.761373 Client: fs_client_1 received a Single RESULT with id: fs_client_1:23


2026-02-24T16:08:33.813470 Client: fs_client_1 received a Single RESULT with id: fs_client_1:24
2026-02-24T16:08:33.907531 Client: fs_client_1 received a Single RESULT with id: fs_client_1:25
2026-02-24T16:08:33.918018 Client: fs_client_1 received a Single RESULT with id: fs_client_1:26
2026-02-24T16:08:33.978602 Client: fs_client_1 received a Single RESULT with id: fs_client_1:27
2026-02-24T16:08:34.017427 Client: fs_client_1 received a Single RESULT with id: fs_client_1:28


['fs_client_1:29']

In [23]:
[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'PENDING']

In [24]:
client.wait_responses()

2026-02-24T16:08:44.103656 Client: fs_client_1 received a Single RESULT with id: fs_client_1:29


[]

In [25]:
# AsynResult.status updates information

[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

In [26]:
try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [27]:
client._update_cache_with_all_available_responses()

In [28]:
client.pending

{}

In [29]:
[f.get() for f in fs]

[0.9766705074657394,
 0.5874971418943278,
 1.0681324410429465,
 [0.30252028666243946, 0.009397501679387243],
 [0.10256825028452432, 0.23266945623182167],
 {'a': 0.19242795927761092, 'b': [0.19242795927761092, 0.4842972791709561]},
 [0.46350235878312995, 0.972627751404377],
 [0.24570457710761962, 0.553547538112171],
 (0.77552703587268, 0.2392960333276557),
 0.5317368664587147,
 None]

In [30]:
fs = client.rpc_batch_async(tp)

2026-02-24T16:08:44.266655 Client: fs_client_1 sent batch request with 11 requests to queue: requests_cola_1


In [31]:
[f.get() for f in fs]

2026-02-24T16:08:54.463219 Client: fs_client_1 received a Batch Response with 11 items
2026-02-24T16:08:54.463918 Client: fs_client_1 processed a RESULT with id: fs_client_1:30 from BATCH response
2026-02-24T16:08:54.464458 Client: fs_client_1 processed a RESULT with id: fs_client_1:31 from BATCH response
2026-02-24T16:08:54.464884 Client: fs_client_1 processed a RESULT with id: fs_client_1:32 from BATCH response
2026-02-24T16:08:54.465392 Client: fs_client_1 processed a RESULT with id: fs_client_1:33 from BATCH response
2026-02-24T16:08:54.466176 Client: fs_client_1 processed a RESULT with id: fs_client_1:34 from BATCH response
2026-02-24T16:08:54.466569 Client: fs_client_1 processed a RESULT with id: fs_client_1:35 from BATCH response
2026-02-24T16:08:54.467061 Client: fs_client_1 processed a RESULT with id: fs_client_1:36 from BATCH response
2026-02-24T16:08:54.467625 Client: fs_client_1 processed a RESULT with id: fs_client_1:37 from BATCH response
2026-02-24T16:08:54.467978 Client

[0.9766705074657394,
 0.5874971418943278,
 1.0681324410429465,
 [0.30252028666243946, 0.009397501679387243],
 [0.10256825028452432, 0.23266945623182167],
 {'a': 0.19242795927761092, 'b': [0.19242795927761092, 0.4842972791709561]},
 [0.46350235878312995, 0.972627751404377],
 [0.24570457710761962, 0.553547538112171],
 (0.77552703587268, 0.2392960333276557),
 0.5317368664587147,
 None]

In [32]:
client.rpc_batch_sync(tp)

2026-02-24T16:08:54.512699 Client: fs_client_1 sent batch request with 11 requests to queue: requests_cola_1


2026-02-24T16:09:04.728313 Client: fs_client_1 received a Batch Response with 11 items
2026-02-24T16:09:04.728927 Client: fs_client_1 processed a RESULT with id: fs_client_1:41 from BATCH response
2026-02-24T16:09:04.729371 Client: fs_client_1 processed a RESULT with id: fs_client_1:42 from BATCH response
2026-02-24T16:09:04.729739 Client: fs_client_1 processed a RESULT with id: fs_client_1:43 from BATCH response
2026-02-24T16:09:04.730187 Client: fs_client_1 processed a RESULT with id: fs_client_1:44 from BATCH response
2026-02-24T16:09:04.730643 Client: fs_client_1 processed a RESULT with id: fs_client_1:45 from BATCH response
2026-02-24T16:09:04.731078 Client: fs_client_1 processed a RESULT with id: fs_client_1:46 from BATCH response
2026-02-24T16:09:04.731522 Client: fs_client_1 processed a RESULT with id: fs_client_1:47 from BATCH response
2026-02-24T16:09:04.731913 Client: fs_client_1 processed a RESULT with id: fs_client_1:48 from BATCH response
2026-02-24T16:09:04.732260 Client

[0.9766705074657394,
 0.5874971418943278,
 1.0681324410429465,
 [0.30252028666243946, 0.009397501679387243],
 [0.10256825028452432, 0.23266945623182167],
 {'a': 0.19242795927761092, 'b': [0.19242795927761092, 0.4842972791709561]},
 [0.46350235878312995, 0.972627751404377],
 [0.24570457710761962, 0.553547538112171],
 (0.77552703587268, 0.2392960333276557),
 0.5317368664587147,
 None]

In [33]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)

('mul', [0.9767288226447031, 0.0842276551086315], {})
('div', [0.8497580841135584, 0.4609202054134174], {})
('div', [0.38789525118524515, 0.991239131923525], {})
('div', [0.0007723779158997912, 0.5347434356292531], {})
('mul', [0.45500416508841557, 0.47586162108866603], {})
('mul', [0.8611444306922785, 0.2835750292704785], {})
('fake', [0.7769607389966463, 0.406619473611719], {})
('div', [0.9106725188820237, 0.5295617864697916], {})
('mul', [0.5567772594591246, 0.42011929132555936], {})
('add', [0.36189342377371114, 0.41349464471349184], {})


In [34]:
tp

[('mul', [0.9767288226447031, 0.0842276551086315], {}),
 ('div', [0.8497580841135584, 0.4609202054134174], {}),
 ('div', [0.38789525118524515, 0.991239131923525], {}),
 ('div', [0.0007723779158997912, 0.5347434356292531], {}),
 ('mul', [0.45500416508841557, 0.47586162108866603], {}),
 ('mul', [0.8611444306922785, 0.2835750292704785], {}),
 ('fake', [0.7769607389966463, 0.406619473611719], {}),
 ('div', [0.9106725188820237, 0.5295617864697916], {}),
 ('mul', [0.5567772594591246, 0.42011929132555936], {}),
 ('add', [0.36189342377371114, 0.41349464471349184], {})]

In [35]:
client.check_registry ="never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

2026-02-24T16:09:04.819192 Client: fs_client_1 sent batch request with 10 requests to queue: requests_cola_1


In [36]:
[f.safe_get() for f in fs]

2026-02-24T16:09:04.961834 Client: fs_client_1 received a Batch Response with 10 items
2026-02-24T16:09:04.962903 Client: fs_client_1 processed a RESULT with id: fs_client_1:52 from BATCH response
2026-02-24T16:09:04.963601 Client: fs_client_1 processed a RESULT with id: fs_client_1:53 from BATCH response
2026-02-24T16:09:04.964220 Client: fs_client_1 processed a RESULT with id: fs_client_1:54 from BATCH response
2026-02-24T16:09:04.964740 Client: fs_client_1 processed a RESULT with id: fs_client_1:55 from BATCH response
2026-02-24T16:09:04.965222 Client: fs_client_1 processed a RESULT with id: fs_client_1:56 from BATCH response
2026-02-24T16:09:04.965721 Client: fs_client_1 processed a RESULT with id: fs_client_1:57 from BATCH response
2026-02-24T16:09:04.966283 Client: fs_client_1 processed a ERROR with id: fs_client_1:58 from BATCH response
2026-02-24T16:09:04.966701 Client: fs_client_1 processed a RESULT with id: fs_client_1:59 from BATCH response
2026-02-24T16:09:04.967134 Client:

[0.08226757840837776,
 1.8436121353182533,
 0.3913235854929622,
 0.00144438970997541,
 0.21651901960106845,
 0.24419905713967244,
 None,
 1.7196718912684843,
 0.2339128676701545,
 0.775388068487203]

In [37]:
client.responses

{}

In [38]:
client.check_registry ="cache"

In [39]:
try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [40]:
try:    
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [41]:
client.check_registry="never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

2026-02-24T16:09:05.100802 Client: fs_client_1 sent request with id: fs_client_1:62 to queue: requests_cola_1


In [42]:
try:
    x.get()
except Exception as e:
    print(e)

2026-02-24T16:09:05.293541 Client: fs_client_1 received a Single ERROR with id: fs_client_1:62


Error -32601 : The method does not exist/is not available.

 


In [43]:
y = client.rpc_async("add", [1, 0])

2026-02-24T16:09:05.326633 Client: fs_client_1 sent request with id: fs_client_1:63 to queue: requests_cola_1


In [44]:
y.get(5)

2026-02-24T16:09:05.560499 Client: fs_client_1 received a Single RESULT with id: fs_client_1:63


1

In [45]:
def f(x,y): return x + y

client.check_registry = "never" # "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn 
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

2026-02-24T16:09:05.595436 Client: fs_client_1 sent request with id: fs_client_1:64 to queue: requests_cola_1
2026-02-24T16:09:05.696570 Client: fs_client_1 received a Single ERROR with id: fs_client_1:64


Error -32601 : The method does not exist/is not available.

 


In [46]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

2026-02-24T16:09:05.752075 Client: fs_client_1 sent request with id: fs_client_1:65 to queue: requests_py_eval


In [47]:
y.get()

2026-02-24T16:09:05.911955 Client: fs_client_1 received a Single RESULT with id: fs_client_1:65


3.0

In [48]:
[k for k in client.responses.keys()]

[]

In [49]:
client.clean_used()

In [50]:
[k for k in client.responses.keys()]

[]

In [51]:
resultados

[0.9766705074657394,
 0.5874971418943278,
 1.0681324410429465,
 [0.30252028666243946, 0.009397501679387243],
 [0.10256825028452432, 0.23266945623182167],
 {'a': 0.19242795927761092, 'b': [0.19242795927761092, 0.4842972791709561]},
 [0.46350235878312995, 0.972627751404377],
 [0.24570457710761962, 0.553547538112171],
 (0.77552703587268, 0.2392960333276557),
 0.5317368664587147,
 None]

In [52]:
client.rpc_sync_fn(print, ["hola"])

2026-02-24T16:09:06.090055 Client: fs_client_1 sent request with id: fs_client_1:66 to queue: requests_py_eval
2026-02-24T16:09:06.251791 Client: fs_client_1 received a Single RESULT with id: fs_client_1:66


In [53]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("sleep", "sleep"))
    t = (fn, [15], {})
    print(t)
    tp.append(t)

('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})


In [54]:
client.check_registry="never"
fs = client.rpc_multi_async(tp, retry=True)

2026-02-24T16:09:06.778852 Client: fs_client_1 sent request with id: fs_client_1:67 to queue: requests_cola_1
2026-02-24T16:09:07.288872 Client: fs_client_1 sent request with id: fs_client_1:68 to queue: requests_cola_1
2026-02-24T16:09:07.310558 Client: fs_client_1 sent request with id: fs_client_1:69 to queue: requests_cola_1
2026-02-24T16:09:07.328064 Client: fs_client_1 sent request with id: fs_client_1:70 to queue: requests_cola_1
2026-02-24T16:09:07.345171 Client: fs_client_1 sent request with id: fs_client_1:71 to queue: requests_cola_1
2026-02-24T16:09:07.386197 Client: fs_client_1 sent request with id: fs_client_1:72 to queue: requests_cola_1
2026-02-24T16:09:07.416172 Client: fs_client_1 sent request with id: fs_client_1:73 to queue: requests_cola_1
2026-02-24T16:09:07.442163 Client: fs_client_1 sent request with id: fs_client_1:74 to queue: requests_cola_1
2026-02-24T16:09:07.463923 Client: fs_client_1 sent request with id: fs_client_1:75 to queue: requests_cola_1
2026-02-24

In [55]:
from time import time

def gather(arl, timeout=None, step=5, max_dts=0):
    # Update to reset delta times
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
    pending_ids = [ar.id for ar in pending_ars]
    if fs[0]._client.wait_responses(pending_ids, timeout=0) == []:
        return

    N = len(arl)

    max_dts = max_dts

    t_0 = time()

    i = 0
    # ok() and failed() don't update.
    # pending() updates, every time is called, all the ids 
    # that have a response available in the queue 
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())]
    pending_ars0= pending_ars
    pending_ids = [ar.id for ar in pending_ars]
    t_prev = time()
    n_pending_prev = len(pending_ars)
    n_pending_0 = n_pending_prev    
    while (time() - t_0) <= timeout:
        try:
            if fs[0]._client.wait_responses(pending_ids, timeout=5) == []:
                return
        except TimeoutError:
            pass
        dts = [ar.finished_time - t_0 for ar in pending_ars0 if (ar.ok() or ar.failed())]
        if len(dts)>0:
            #max_dts = max(max_dts, max(dts))
            max_dts = max(dts)    
        pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
        pending_ids = [ar.id for ar in pending_ars]
        if len(pending_ars) < n_pending_prev:
            t_prev = time()
            n_pending_prev = len(pending_ars)  

        avg =  (time()-t_0)/(n_pending_0 -len(pending_ars)) if (n_pending_0 -len(pending_ars))>0 else 0

        print(f"{i}: seconds {time()-t_0}s, AR recovered {N-len(pending_ars)}, \
                AR left {len(pending_ars)}, max delta {max_dts}, avg {avg}")
        i += 0


In [56]:
gather(fs, 30, 5)

0: seconds 5.2058868408203125s, AR recovered 0,                 AR left 10, max delta 0, avg 0
0: seconds 10.355451345443726s, AR recovered 0,                 AR left 10, max delta 0, avg 0


2026-02-24T16:09:21.916898 Client: fs_client_1 received a Single RESULT with id: fs_client_1:67


0: seconds 15.506098985671997s, AR recovered 1,                 AR left 9, max delta 14.360270023345947, avg 15.506097078323364
0: seconds 20.633108139038086s, AR recovered 1,                 AR left 9, max delta 14.360270023345947, avg 20.633105278015137
0: seconds 25.79712438583374s, AR recovered 1,                 AR left 9, max delta 14.360270023345947, avg 25.797122478485107


2026-02-24T16:09:36.963247 Client: fs_client_1 received a Single RESULT with id: fs_client_1:68


0: seconds 30.949328660964966s, AR recovered 2,                 AR left 8, max delta 29.40661931037903, avg 15.474663376808167


In [57]:
[f.status for f in fs]

['OK',
 'OK',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING']

In [58]:
[(time() if f.finished_time is None else f.finished_time) -f.creation_time for f in fs]

[15.136986255645752,
 29.673134088516235,
 31.23414921760559,
 31.21694016456604,
 31.199662923812866,
 31.157079696655273,
 31.125852346420288,
 31.10293960571289,
 31.080950498580933,
 31.05947494506836]

In [59]:
[f.finished_time for f in fs]

[1771945761.9168944,
 1771945776.9632437,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [60]:
client.pending

{'fs_client_1:69': 1771945747.3117115,
 'fs_client_1:70': 1771945747.3289223,
 'fs_client_1:71': 1771945747.3462,
 'fs_client_1:72': 1771945747.3887813,
 'fs_client_1:73': 1771945747.4200077,
 'fs_client_1:74': 1771945747.4429235,
 'fs_client_1:75': 1771945747.464913,
 'fs_client_1:76': 1771945747.4863882}

In [61]:
fs[3].status

'PENDING'

In [62]:
fs[0].retry()

2026-02-24T16:09:38.681026 Client: fs_client_1 Not Retrying: Response to Request with id: fs_client_1:67 already received.


In [63]:
[f.retry() for f in fs if ok is not None]

NameError: name 'ok' is not defined

In [ ]:
import datetime

In [ ]:
datetime.datetime.now().isoformat()

'2025-10-29T18:22:36.906351'

In [ ]:
from datetime import datetime

In [ ]:
client.check_registry="always"
client.all_queues_for_method("info")

['requests_fs_server_1', 'requests_fs_server_2']

In [ ]:
client.connector.all_queues_for_method("info")

['requests_fs_server_1', 'requests_fs_server_2']

In [ ]:
client.update_registry()

registry_lock set
registry_lock unset


In [ ]:
client.check_registry="Never"
client.all_queues_for_method("hola")

['requests_cola_1']

In [ ]:
client.check_registry="always"
a = client.rpc_async("info")

2025-10-29T18:22:37.090210 Client: fs_client_2 sent request with id: fs_client_2:77 to queue: requests_fs_server_2


In [ ]:
client.rpc_sync("div", [2,1], timeout=10)

2025-10-29T18:22:37.118667 Client: fs_client_2 sent request with id: fs_client_2:78 to queue: requests_cola_1


2025-10-29T18:22:37.260453 Client: fs_client_2 received a Single RESULT with id: fs_client_2:77
2025-10-29T18:22:37.273416 Client: fs_client_2 received a Single RESULT with id: fs_client_2:78


2.0

In [ ]:
x = a.get()

In [ ]:
x[1]["requests_cola_1"]["add"](1,2)

3

In [ ]:
client.registry

{'add': ['requests_cola_1'],
 'dic': ['requests_cola_1'],
 'div': ['requests_cola_1'],
 'eval_py_function': ['requests_py_eval'],
 'info': ['requests_fs_server_1', 'requests_fs_server_2'],
 'lista': ['requests_cola_1'],
 'mul': ['requests_cola_1'],
 'sleep': ['requests_cola_1'],
 'tupla': ['requests_cola_1']}

In [ ]:
def registry_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue, methods in x[1].items():
        for method in methods:
            if method in registry:
                registry[method].append(queue)
            else:
                registry[method] = [queue]
    return registry 

def queues_workers_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue in x[1]:
        if queue in registry:
            registry[queue].append(worker_id)
        else:
            registry[queue] = [worker_id]
    return registry 

def all_workers_for_queue():
    # all requests queues for "info" method
    # One queue per worker

    rr = {}
    qq = {}
    all_worker_queues = client.connector.all_queues_for_method("info")
    for worker in all_worker_queues:
        try:
            x = client.rpc_sync("info", timeout=10)
        except TimeoutError:
            x = None
        
        if x is not None:
            for method, qs in registry_one_step(x).items():
                if method not in rr:
                    rr[method] = qs
                else:
                    rr[method] += qs
            for q, ws in queues_workers_one_step(x).items():
                if q not in qq:
                    qq[q] = ws
                else:
                    qq[q] += ws
    return rr, qq
            

In [ ]:
registry_one_step(x)

{'info': ['requests_fs_server_2'],
 'add': ['requests_cola_1'],
 'mul': ['requests_cola_1'],
 'div': ['requests_cola_1'],
 'lista': ['requests_cola_1'],
 'tupla': ['requests_cola_1'],
 'dic': ['requests_cola_1'],
 'sleep': ['requests_cola_1'],
 'hola': ['requests_cola_2'],
 'eval_py_function': ['requests_py_eval']}

In [ ]:
queues_workers_one_step(x)

{'requests_fs_server_2': ['fs_server_2'],
 'requests_cola_1': ['fs_server_2'],
 'requests_cola_2': ['fs_server_2'],
 'requests_py_eval': ['fs_server_2']}

In [ ]:
r, q = all_workers_for_queue()

2025-10-29T18:22:37.461400 Client: fs_client_2 sent request with id: fs_client_2:79 to queue: requests_fs_server_1
2025-10-29T18:22:37.661109 Client: fs_client_2 received a Single RESULT with id: fs_client_2:79


2025-10-29T18:22:37.681071 Client: fs_client_2 sent request with id: fs_client_2:80 to queue: requests_fs_server_2
2025-10-29T18:22:37.924273 Client: fs_client_2 received a Single RESULT with id: fs_client_2:80


In [ ]:
r

{'info': ['requests_fs_server_1', 'requests_fs_server_2'],
 'add': ['requests_cola_1', 'requests_cola_1'],
 'mul': ['requests_cola_1', 'requests_cola_1'],
 'div': ['requests_cola_1', 'requests_cola_1'],
 'lista': ['requests_cola_1', 'requests_cola_1'],
 'tupla': ['requests_cola_1', 'requests_cola_1'],
 'dic': ['requests_cola_1', 'requests_cola_1'],
 'sleep': ['requests_cola_1', 'requests_cola_1'],
 'hola': ['requests_cola_2', 'requests_cola_2'],
 'eval_py_function': ['requests_py_eval', 'requests_py_eval']}

In [ ]:
q

{'requests_fs_server_1': ['fs_server_1'],
 'requests_cola_1': ['fs_server_1', 'fs_server_2'],
 'requests_cola_2': ['fs_server_1', 'fs_server_2'],
 'requests_py_eval': ['fs_server_1', 'fs_server_2'],
 'requests_fs_server_2': ['fs_server_2']}

In [ ]:
t = (1,2,3,4)

In [ ]:
t[:6]

(1, 2, 3, 4)

In [ ]:
t[:5]

(1, 2, 3, 4)